# 03 A/B测试统计分析\n\n## 分析目标\n- 模拟A/B实验：按user_id奇偶分组，对比两组转化率差异\n- 进行样本量检验与统计假设检验\n- 计算p值、置信区间与效应量\n- 给出实验结论与业务建议

In [ ]:
import pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\nimport warnings\nimport os\nfrom datetime import datetime, timedelta\nfrom scipy import stats\n\nwarnings.filterwarnings('ignore')\nplt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']\nplt.rcParams['axes.unicode_minus'] = False\nsns.set_style('whitegrid')\n\nIMG_DIR = '../images'\nos.makedirs(IMG_DIR, exist_ok=True)\n\nprint('环境初始化完成')

## 1. 数据加载与实验分组\n\n**分组规则**：user_id为奇数 → 实验组（B组，新策略）；user_id为偶数 → 对照组（A组，原策略）。\n**指标定义**：转化率 = 有购买行为的用户数 / 总用户数。

In [ ]:
DATA_PATH = '../data/processed/user_behavior_cleaned.csv'\n\nif os.path.exists(DATA_PATH):\n    df = pd.read_csv(DATA_PATH, parse_dates=['datetime', 'date'])\n    print('已加载真实数据')\nelse:\n    np.random.seed(42)\n    n = 100000\n    users = np.random.choice(range(1000, 9000), n)\n    items = np.random.choice(range(100000, 900000), n)\n    cates = np.random.choice(range(100, 1200), n)\n    behaviors = np.random.choice(['pv', 'buy', 'cart', 'fav'], n, p=[0.70, 0.05, 0.15, 0.10])\n    start = datetime(2017, 11, 25)\n    deltas = np.random.randint(0, 9*24*3600, n)\n    datetimes = [start + timedelta(seconds=int(s)) for s in deltas]\n    df = pd.DataFrame({\n        'user_id': users, 'item_id': items, 'category_id': cates,\n        'behavior_type': behaviors, 'timestamp': [int(d.timestamp()) for d in datetimes],\n        'datetime': datetimes,\n    })\n    df['date'] = df['datetime'].dt.date\n    df['hour'] = df['datetime'].dt.hour\n    df['day_of_week'] = df['datetime'].dt.dayofweek\n    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)\n    def time_period(h):\n        if 0 <= h < 6: return '凌晨'\n        elif 6 <= h < 12: return '上午'\n        elif 12 <= h < 18: return '下午'\n        else: return '晚上'\n    df['time_period'] = df['hour'].apply(time_period)\n    print('已生成模拟数据')\n\n# 实验分组\ndf['group'] = df['user_id'].apply(lambda x: 'B' if x % 2 == 1 else 'A')\n\n# 用户级别聚合：是否购买\nuser_buy = df.groupby('user_id').agg(\n    group=('group', 'first'),\n    is_buy=('behavior_type', lambda x: int((x == 'buy').any()))\n).reset_index()\n\nprint('分组后用户数:')\nprint(user_buy['group'].value_counts())

## 2. 转化率计算

In [ ]:
summary = user_buy.groupby('group').agg(\n    total_users=('user_id', 'count'),\n    buyers=('is_buy', 'sum')\n).reset_index()\nsummary['conversion_rate'] = summary['buyers'] / summary['total_users']\n\nprint(summary)

## 3. 样本量检验\n\n**检验思路**：使用两比例差异检验的样本量公式，确保实验有足够的统计功效（power=0.8，显著性水平α=0.05）。\n若实际样本量大于最小样本量，则认为样本充足。

In [ ]:
def required_sample_size(p1: float, p2: float, alpha: float = 0.05, power: float = 0.8) -> float:\n    """\n    计算两比例检验所需的最小样本量（每组）。\n    使用正态近似公式：n = (Z_alpha/2 + Z_beta)^2 * (p1*(1-p1) + p2*(1-p2)) / (p1-p2)^2\n    """\n    z_alpha = stats.norm.ppf(1 - alpha/2)\n    z_beta = stats.norm.ppf(power)\n    se = np.sqrt(p1*(1-p1) + p2*(1-p2))\n    delta = abs(p1 - p2)\n    if delta == 0:\n        return np.inf\n    n = ((z_alpha + z_beta) * se / delta) ** 2\n    return n\n\np_a = summary.loc[summary['group'] == 'A', 'conversion_rate'].values[0]\np_b = summary.loc[summary['group'] == 'B', 'conversion_rate'].values[0]\nn_a = summary.loc[summary['group'] == 'A', 'total_users'].values[0]\nn_b = summary.loc[summary['group'] == 'B', 'total_users'].values[0]\n\nn_required = required_sample_size(p_a, p_b)\nprint(f'对照组转化率: {p_a:.4f}')\nprint(f'实验组转化率: {p_b:.4f}')\nprint(f'每组最小样本量要求: {n_required:.0f}')\nprint(f'对照组实际样本量: {n_a}')\nprint(f'实验组实际样本量: {n_b}')\nprint('样本量是否充足:', '是' if min(n_a, n_b) >= n_required else '否')

## 4. 假设检验\n\n### 4.1 卡方检验\n检验两组转化率是否存在显著差异。

In [ ]:
# 构建列联表\ncontingency = pd.crosstab(user_buy['group'], user_buy['is_buy'])\nprint('列联表:')\nprint(contingency)\n\nchi2, p_chi2, dof, expected = stats.chi2_contingency(contingency)\nprint(f'\\n卡方值: {chi2:.4f}')\nprint(f'p值: {p_chi2:.4f}')\nprint(f'自由度: {dof}')

### 4.2 Z检验（两比例检验）

In [ ]:
def two_proportion_z_test(x1, n1, x2, n2):\n    """\n    两比例Z检验。\n    返回: z统计量, p值（双侧）\n    """\n    p1 = x1 / n1\n    p2 = x2 / n2\n    p_pool = (x1 + x2) / (n1 + n2)\n    se = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))\n    if se == 0:\n        return 0.0, 1.0\n    z = (p1 - p2) / se\n    p_value = 2 * (1 - stats.norm.cdf(abs(z)))\n    return z, p_value\n\nx_a = summary.loc[summary['group'] == 'A', 'buyers'].values[0]\nx_b = summary.loc[summary['group'] == 'B', 'buyers'].values[0]\n\nz_stat, p_z = two_proportion_z_test(x_a, n_a, x_b, n_b)\nprint(f'Z统计量: {z_stat:.4f}')\nprint(f'p值: {p_z:.4f}')

## 5. 置信区间与效应量

In [ ]:
def proportion_ci(x, n, alpha=0.05):\n    """计算比例的置信区间（正态近似）。"""\n    p = x / n\n    z = stats.norm.ppf(1 - alpha/2)\n    se = np.sqrt(p * (1 - p) / n)\n    return p - z*se, p + z*se\n\nci_a = proportion_ci(x_a, n_a)\nci_b = proportion_ci(x_b, n_b)\n\n# 效应量：Cohen's h\ndef cohens_h(p1, p2):\n    return 2 * (np.arcsin(np.sqrt(p1)) - np.arcsin(np.sqrt(p2)))\n\neffect = cohens_h(p_a, p_b)\n\nprint(f'对照组转化率 95% CI: [{ci_a[0]:.4f}, {ci_a[1]:.4f}]')\nprint(f'实验组转化率 95% CI: [{ci_b[0]:.4f}, {ci_b[1]:.4f}]')\nprint(f'效应量 Cohen\'s h: {effect:.4f}')

## 6. 可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# 转化率对比柱状图\nsns.barplot(data=summary, x='group', y='conversion_rate', palette='coolwarm', ax=axes[0])\naxes[0].set_title('A/B组转化率对比', fontsize=14)\naxes[0].set_xlabel('分组')\naxes[0].set_ylabel('转化率')\nfor i, v in enumerate(summary['conversion_rate']):\n    axes[0].text(i, v + max(summary['conversion_rate'])*0.01, f'{v:.3f}', ha='center', va='bottom')\n\n# 置信区间图\ngroups = ['A', 'B']\nmeans = [p_a, p_b]\nerrors = [[p_a - ci_a[0], p_b - ci_b[0]], [ci_a[1] - p_a, ci_b[1] - p_b]]\naxes[1].errorbar(groups, means, yerr=errors, fmt='o', capsize=8, capthick=2, markersize=8, color='steelblue')\naxes[1].set_title('转化率 95% 置信区间', fontsize=14)\naxes[1].set_xlabel('分组')\naxes[1].set_ylabel('转化率')\naxes[1].set_ylim(0, max(means)*1.5)\n\nplt.tight_layout()\nplt.savefig(f'{IMG_DIR}/03_ab_test_results.png', dpi=150, bbox_inches='tight')\nplt.show()

## 7. 实验结论与建议\n\n1. **统计显著性**：若p值 < 0.05，则两组转化率差异显著；否则差异可能由随机波动导致。\n2. **业务显著性**：即使统计显著，也需评估效应量（Cohen's h）。小效应量（|h|<0.2）在实际业务中意义有限。\n3. **样本量**：若样本不足，应延长实验周期或扩大流量，避免犯第二类错误。\n4. **建议**：实验组若显著优于对照组且效应量中等以上，可逐步全量上线；若差异不显著，需回溯实验设计（如分组是否随机、策略是否真正触达用户）。